# Pembersihan Dataset Emosi Ataraxia

Notebook ini membersihkan data mentah `dataset_mentah_ataraxia.csv` dari awal: mulai dari eksplorasi masalah, penentuan aturan pembersihan, penerapan, sampai pengecekan kualitas hasil.

**Alur kerja**
1. Load data mentah
2. Eksplorasi masalah (label, teks, duplikat)
3. Pembersihan label
4. Pembersihan teks
5. Pembuangan baris bermasalah dan duplikat
6. Pengecekan kualitas
7. Simpan hasil

In [ ]:
import os
import re
import unicodedata

import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

RAW_PATH = "dataset_mentah_ataraxia.csv"
OUT_PATH = "data/dataset_clean_ataraxia.csv"

# Label emosi yang dianggap sah
VALID_LABELS = {"senang", "sedih", "marah", "takut", "jijik", "kaget", "cape", "netral"}

## 1. Load data mentah

In [ ]:
df_raw = pd.read_csv(RAW_PATH, dtype=str)
print("Shape  :", df_raw.shape)
print("Kolom  :", df_raw.columns.tolist())
df_raw.head(10)

## 2. Eksplorasi masalah

### 2.1 Nilai kosong

In [ ]:
df_raw.isna().sum()

### 2.2 Label

In [ ]:
print("Jumlah variasi label unik:", df_raw["label"].nunique(dropna=True))
df_raw["label"].value_counts(dropna=False)

Label yang sama ditulis dengan banyak variasi (`senang`, `Senang`, `SENANG`, ` senang`). Selain itu ada label di luar 8 kelas emosi (`bahagia`, `unknown`, `test`, `???`, `3`) dan label kosong.

In [ ]:
label_norm = df_raw["label"].str.strip().str.lower()
print("Label tidak valid setelah normalisasi:")
label_norm[~label_norm.isin(VALID_LABELS)].value_counts(dropna=False)

### 2.3 Teks

In [ ]:
teks = df_raw["text"].fillna("")

masalah = {
    "URL"                    : teks.str.contains(r"https?://\S+"),
    "Mention (@user)"        : teks.str.contains(r"@\w+"),
    "Karakter zero-width"    : teks.str.contains("[\u200b-\u200f\u2060\ufeff]"),
    "Tab"                    : teks.str.contains("\t"),
    "Spasi ganda"            : teks.str.contains(r" {2,}"),
    "Spasi di awal/akhir"    : teks.str.contains(r"^\s|\s$"),
    "Mengandung huruf besar" : teks.str.contains(r"[A-Z]"),
    "Teks kosong/spasi saja" : teks.str.strip() == "",
}
pd.Series({k: int(v.sum()) for k, v in masalah.items()}, name="jumlah baris").to_frame()

### 2.4 Contoh teks bermasalah

In [ ]:
for nama in ["URL", "Mention (@user)", "Tab", "Mengandung huruf besar"]:
    print(f"--- {nama}")
    for t in df_raw.loc[masalah[nama].values, "text"].head(3):
        print(repr(t))

### 2.5 Duplikat

Duplikat dicek setelah teks dinormalisasi, karena dua kalimat yang sama bisa tampak berbeda hanya karena huruf besar, spasi, atau tambahan URL.

## 3. Fungsi pembersihan

**Label:** buang spasi tepi, ubah ke huruf kecil.

**Teks** (urutan penting):
1. Normalisasi unicode (NFKC)
2. Hapus karakter zero-width
3. Hapus URL
4. Hapus mention
5. Ganti tab, newline, dan spasi ganda dengan satu spasi, lalu buang spasi tepi
6. Ubah ke huruf kecil


In [ ]:
ZERO_WIDTH = re.compile(r"[\u200b-\u200f\u2060\ufeff]")
URL        = re.compile(r"https?://\S+")
MENTION    = re.compile(r"@\w+")
WHITESPACE = re.compile(r"\s+")


def clean_label(value):
    if pd.isna(value):
        return None
    return value.strip().lower()


def clean_text(value):
    if pd.isna(value):
        return None
    value = unicodedata.normalize("NFKC", value)
    value = ZERO_WIDTH.sub("", value)
    value = URL.sub("", value)
    value = MENTION.sub("", value)
    value = WHITESPACE.sub(" ", value).strip()
    return value.lower()

In [ ]:
# Uji cepat pada contoh
contoh = [
    "  TEMAN  SEBELAH RUMAHNYA RAMAH. SENANG SEKALI!",
    "sahabatku pindah jauh, aku sedih banget.    http://t.co/e8aighf",
    "@rina Luar rumah terdengar suara keras. aku takut sekali!",
    "Saya capek. angk\u200bat barang berat.",
    "Microwave\tada makanan. rusak! jijik!",
]
for c in contoh:
    print(repr(c), "->", repr(clean_text(c)))

## 4. Terapkan pembersihan

In [ ]:
df = df_raw.copy()
df["label"] = df["label"].map(clean_label)
df["text"]  = df["text"].map(clean_text)

df[["label", "text"]].head(10)

## 5. Buang baris bermasalah

Setiap tahap dicatat jumlah barisnya agar proses bisa ditelusuri.

In [ ]:
log = [("Data mentah", len(df))]

# a. label tidak valid atau kosong
df = df[df["label"].isin(VALID_LABELS)]
log.append(("Setelah buang label tidak valid/kosong", len(df)))

# b. teks kosong
df = df[df["text"].notna() & (df["text"] != "")]
log.append(("Setelah buang teks kosong", len(df)))

# c. duplikat teks (dipertahankan kemunculan pertama)
df = df.drop_duplicates(subset="text", keep="first")
log.append(("Setelah hapus duplikat", len(df)))

df = df.reset_index(drop=True)

ringkasan = pd.DataFrame(log, columns=["tahap", "jumlah baris"])
ringkasan["baris dibuang"] = -ringkasan["jumlah baris"].diff().fillna(0).astype(int)
ringkasan

## 6. Pengecekan kualitas

Memastikan tidak ada masalah yang tersisa setelah pembersihan.

In [ ]:
cek = {
    "Nilai kosong"                : int(df[["label", "text"]].isna().sum().sum()),
    "Label di luar 8 kelas"       : int((~df["label"].isin(VALID_LABELS)).sum()),
    "Teks kosong"                 : int((df["text"] == "").sum()),
    "Teks duplikat"               : int(df["text"].duplicated().sum()),
    "Teks dengan label bertentangan": int((df.groupby("text")["label"].nunique() > 1).sum()),
    "Masih ada URL"               : int(df["text"].str.contains(r"https?://").sum()),
    "Masih ada mention"           : int(df["text"].str.contains(r"@\w+").sum()),
    "Masih ada zero-width"        : int(df["text"].str.contains("[\u200b-\u200f\u2060\ufeff]").sum()),
    "Masih ada tab/spasi ganda"   : int(df["text"].str.contains(r"\t| {2,}").sum()),
    "Masih ada spasi tepi"        : int(df["text"].str.contains(r"^\s|\s$").sum()),
    "Masih ada huruf besar"       : int(df["text"].str.contains(r"[A-Z]").sum()),
}
hasil_cek = pd.Series(cek, name="jumlah").to_frame()
display(hasil_cek)

assert (hasil_cek["jumlah"] == 0).all(), "Masih ada masalah yang belum bersih"
print("Semua pengecekan lolos.")

In [ ]:
print("Shape akhir:", df.shape, "\n")
print("Distribusi label:")
df["label"].value_counts()

In [ ]:
panjang = df["text"].str.len()
panjang.describe().round(2).to_frame("panjang teks (karakter)")

In [ ]:
# Perbandingan sebelum dan sesudah pada baris acak
idx = df_raw.dropna(subset=["text"]).sample(8, random_state=7).index
pd.DataFrame({
    "label_mentah": df_raw.loc[idx, "label"],
    "teks_mentah" : df_raw.loc[idx, "text"],
    "teks_bersih" : df_raw.loc[idx, "text"].map(clean_text),
})

## 7. Simpan hasil

In [ ]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
df[["label", "text"]].to_csv(OUT_PATH, index=False)
print(f"Tersimpan ke {OUT_PATH} ({len(df):,} baris)")

## Ringkasan

- Label diseragamkan menjadi 8 kelas dan baris berlabel tidak valid atau kosong dibuang.
- Teks dibersihkan dari URL, mention, karakter tak terlihat, tab, dan spasi berlebih, lalu diubah ke huruf kecil.
- Baris dengan teks kosong dan teks duplikat dibuang.
- Hasil akhir disimpan ke `data/dataset_clean_ataraxia.csv` dan siap dipakai oleh aplikasi Streamlit.